# Popular Bookstore — Trade-In Demographic Analytics
**IS215 Digital Business Transformation**

---

## What this notebook does

Popular collects trade-in data every day (genre, condition, timing, outlet) but only uses it to calculate credit value. We think there's more to it.

The idea: from just one trade-in, we can figure out *who* the customer is — a parent buying for their kid, a student done with exams, a casual reader, etc. No survey needed, no login needed. Just the book details.

We got this idea from how Amazon famously predicted a customer was pregnant just from her purchase patterns. Popular already has the data — they're just not using it.

---

## The 5 Personas we defined

We came up with these based on who actually walks into Popular and trades in books:

**1. Parent** — "I just want my child to do well"  
Age 35-45. Trades in children's or assessment books in good condition after the school year. Shops at heartland malls like Tampines, Jurong Point. Price-conscious, buys in bulk.

**2. Student** — "I need to pass my exams"  
Age 13-19. Trades in worn assessment books right after exams. Comes in to get credit for next semester's books. Shows up at outlets near schools.

**3. Bookworm** — "Reading is my hobby"  
Age 20-35. Trades in fiction and manga year-round in good condition. Reads a lot, runs out of shelf space. Frequents city outlets like Bras Basah and Bugis.

**4. Self-Improver** — "I'm investing in myself"  
Age 25-40. Trades in self-help and business books, usually in good condition (bought with good intentions, never finished). City outlets, no seasonal pattern.

**5. Budget Shopper** — "Give me the best deal"  
Any age. Trades in anything in fair/worn condition. Doesn't care about genre — just wants maximum credit. Could show up anywhere.

---

## Models we used
1. **K-Means Clustering** — unsupervised, finds natural groupings
2. **Random Forest Classifier** — supervised, predicts persona from the trade-in data


---
## 1. Imports & Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Consistent style
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f9f9f9'
plt.rcParams['font.family'] = 'DejaVu Sans'
POPULAR_RED = '#E31837'
POPULAR_DARK = '#1a1a1a'

np.random.seed(42)
print('Setup complete.')

---
## 2. Simulated Trade-In Data

We don't have access to Popular's actual data, so we simulated 1000 trade-in transactions.

Each transaction has 4 fields — the same things a cashier would record at the trade-in counter:
- `genre` — what type of book
- `condition` — Good, Fair, or Worn
- `month` — when they came in
- `outlet` — which Popular store

The outlets are all real Popular locations in Singapore. We mapped each outlet to the personas most likely to shop there based on the neighbourhood — e.g. suburban family malls skew toward Parents and Students, city outlets skew toward Bookworms and Self-Improvers.


In [ ]:
N = 1000  # simulated transactions
np.random.seed(42)

# GENRE LIST
genres = [
    'Fiction', "Children's", 'Assessment', 'Self-Help',
    'Business', 'Biography', 'Manga', 'Cooking/Lifestyle'
]

# CONDITIONS (3 only, as per Popular grading)
conditions = ['Good', 'Fair', 'Worn']

# REAL POPULAR OUTLETS + PERSONA SIGNAL
# Each outlet mapped to the personas most likely to trade in there
outlet_persona_map = {
    # Suburban family malls → Parent, Student
    'Tampines Mall':       ['Parent', 'Parent', 'Student', 'Budget Shopper'],
    'Jurong Point':        ['Parent', 'Parent', 'Student', 'Budget Shopper'],
    'Causeway Point':      ['Parent', 'Parent', 'Student', 'Budget Shopper'],
    'Lot One':             ['Parent', 'Student', 'Budget Shopper'],
    'Westgate':            ['Parent', 'Student', 'Budget Shopper'],
    'Bedok Mall':          ['Parent', 'Student', 'Budget Shopper'],
    'White Sands':         ['Parent', 'Student', 'Budget Shopper'],
    'Compass One':         ['Parent', 'Student', 'Budget Shopper'],
    'West Mall':           ['Parent', 'Student', 'Budget Shopper'],
    # Heartland outlets → Parent, Student
    'Toa Payoh HDB Hub':   ['Parent', 'Student', 'Budget Shopper'],
    'Hougang Mall':        ['Parent', 'Student', 'Budget Shopper'],
    'United Square':       ['Parent', 'Student', 'Self-Improver'],
    # Near schools → Student skew
    'Junction 8':          ['Student', 'Parent', 'Budget Shopper'],
    # City/lifestyle outlets → Bookworm, Self-Improver
    'Bras Basah Complex':  ['Bookworm', 'Bookworm', 'Self-Improver', 'Budget Shopper'],
    'Bugis Junction':      ['Bookworm', 'Self-Improver', 'Budget Shopper'],
    'Tiong Bahru Plaza':   ['Bookworm', 'Self-Improver', 'Budget Shopper'],
    'Plaza Singapura':     ['Self-Improver', 'Bookworm', 'Budget Shopper'],
    'Paya Lebar Quarter':  ['Self-Improver', 'Bookworm', 'Budget Shopper'],
    'VivoCity':            ['Bookworm', 'Self-Improver', 'Parent', 'Budget Shopper'],
}

# PERSONA PROFILES
persona_profiles = {
    'Parent': {
        'genres':     ["Children's", "Children's", 'Assessment', 'Assessment', 'Cooking/Lifestyle'],
        'conditions': ['Good', 'Good', 'Fair'],
        'months':     [1, 6, 11, 12],   # new school year + post-exam
        'outlets':    ['Tampines Mall', 'Jurong Point', 'Causeway Point', 'Lot One',
                       'Westgate', 'Bedok Mall', 'White Sands', 'Compass One',
                       'West Mall', 'Toa Payoh HDB Hub', 'Hougang Mall', 'United Square'],
    },
    'Student': {
        'genres':     ['Assessment', 'Assessment', 'Assessment', 'Biography', 'Manga'],
        'conditions': ['Fair', 'Fair', 'Worn'],
        'months':     [1, 6, 11, 12],   # same school calendar as Parent
        'outlets':    ['Junction 8', 'Tampines Mall', 'United Square',
                       'Toa Payoh HDB Hub', 'Bedok Mall', 'Causeway Point'],
    },
    'Bookworm': {
        'genres':     ['Fiction', 'Fiction', 'Fiction', 'Manga', 'Biography'],
        'conditions': ['Good', 'Good', 'Fair'],
        'months':     list(range(1, 13)),  # year-round, no exam pattern
        'outlets':    ['Bras Basah Complex', 'Bugis Junction', 'Tiong Bahru Plaza', 'VivoCity'],
    },
    'Self-Improver': {
        'genres':     ['Self-Help', 'Self-Help', 'Business', 'Biography', 'Cooking/Lifestyle'],
        'conditions': ['Good', 'Good', 'Fair'],
        'months':     list(range(1, 13)),  # year-round
        'outlets':    ['Plaza Singapura', 'Bugis Junction', 'VivoCity',
                       'Tiong Bahru Plaza', 'Paya Lebar Quarter', 'United Square'],
    },
    'Budget Shopper': {
        'genres':     genres,             # any genre
        'conditions': ['Fair', 'Fair', 'Worn', 'Worn'],
        'months':     list(range(1, 13)), # any time
        'outlets':    list(outlet_persona_map.keys()),  # any outlet
    },
}

# GENERATE TRANSACTIONS
rows = []
persona_list  = list(persona_profiles.keys())
persona_probs = [0.25, 0.25, 0.20, 0.18, 0.12]  # realistic SG distribution

for i in range(N):
    persona = np.random.choice(persona_list, p=persona_probs)
    profile = persona_profiles[persona]

    genre     = np.random.choice(profile['genres'])
    condition = np.random.choice(profile['conditions'])
    month     = np.random.choice(profile['months'])
    outlet    = np.random.choice(profile['outlets'])

    rows.append({
        'transaction_id': f'TI-{i+1000}',
        'persona':        persona,
        'genre':          genre,
        'condition':      condition,
        'month':          month,
        'outlet':         outlet,
    })

df = pd.DataFrame(rows)

print(f'Dataset: {len(df)} trade-in transactions')
print(f'\nPersona distribution:')
print(df['persona'].value_counts())
print(f'\nSample transactions:')
df.head(10)

---
## 3. Exploratory Data Analysis
Quick look at how the data is distributed before we start modelling.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Popular Trade-In — Exploratory Analysis', fontsize=16, fontweight='bold', color=POPULAR_DARK)

# Plot 1: Trade-ins by genre
genre_counts = df['genre'].value_counts()
axes[0,0].barh(genre_counts.index, genre_counts.values, color=POPULAR_RED, alpha=0.85)
axes[0,0].set_title('Trade-ins by Genre', fontweight='bold')
axes[0,0].set_xlabel('Number of trade-ins')

# Plot 2: Condition distribution
cond_counts = df['condition'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0,1].pie(cond_counts.values, labels=cond_counts.index, autopct='%1.1f%%',
              colors=colors, startangle=90)
axes[0,1].set_title('Book Condition at Trade-In', fontweight='bold')

# Plot 3: Trade-ins by month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('month').size().reindex(range(1,13), fill_value=0)
axes[1,0].bar(month_names, monthly.values, color=POPULAR_RED, alpha=0.8)
axes[1,0].set_title('Trade-ins by Month (Seasonality)', fontweight='bold')
axes[1,0].set_xlabel('Month')
axes[1,0].set_ylabel('Transactions')

# Plot 4: Trade-ins by persona
persona_counts = df['persona'].value_counts()
bar_colors = [POPULAR_RED, '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
axes[1,1].bar(persona_counts.index, persona_counts.values, color=bar_colors)
axes[1,1].set_title('Trade-ins by Persona', fontweight='bold')
axes[1,1].set_ylabel('Count')
axes[1,1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: Trade-ins spike in Jan and Nov/Dec — aligned with Singapore school calendar')

---
## 4. Feature Engineering
Models need numbers, not text. This converts genre, condition, and outlet into numeric codes.


In [ ]:
# Encode categoricals
le_genre     = LabelEncoder().fit(df['genre'])
le_condition = LabelEncoder().fit(df['condition'])
le_outlet    = LabelEncoder().fit(df['outlet'])
le_persona   = LabelEncoder().fit(df['persona'])

df['genre_enc']     = le_genre.transform(df['genre'])
df['condition_enc'] = le_condition.transform(df['condition'])
df['outlet_enc']    = le_outlet.transform(df['outlet'])
df['persona_enc']   = le_persona.transform(df['persona'])

# Feature matrix — only what Popular can observe at trade-in counter
FEATURES = ['genre_enc', 'condition_enc', 'month', 'outlet_enc']
X = df[FEATURES]
y = df['persona_enc']

print('Features used (all observable at trade-in counter):')
for f in FEATURES:
    print(f'  - {f}')
print(f'\nTarget classes (personas): {list(le_persona.classes_)}')

---
## 5. K-Means Clustering (Unsupervised)
K-Means doesn't know our persona labels — it just looks at the 4 features and tries to find natural groupings on its own. If the clusters it finds line up with our personas, that's a good sign the patterns are real.


In [ ]:
# Elbow method — find optimal number of clusters
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'o-', color=POPULAR_RED, linewidth=2, markersize=8)
plt.axvline(x=5, color='gray', linestyle='--', alpha=0.6, label='Chosen K=5')
plt.title('Elbow Method — Optimal Number of Clusters', fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.legend()
plt.tight_layout()
plt.savefig('elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print('K=5 chosen — matches our 5 persona hypothesis')

In [ ]:
# Fit K-Means with K=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Profile each cluster
cluster_profile = df.groupby('cluster').agg(
    size=('transaction_id', 'count'),
    top_genre=('genre', lambda x: x.mode()[0]),
    top_condition=('condition', lambda x: x.mode()[0]),
    peak_month=('month', lambda x: x.mode()[0]),
    top_outlet=('outlet', lambda x: x.mode()[0]),
    dominant_persona=('persona', lambda x: x.mode()[0])
).reset_index()

print('K-Means Cluster Profiles:')
print(cluster_profile.to_string(index=False))

In [ ]:
# Visualise clusters: genre vs condition, coloured by cluster
fig, ax = plt.subplots(figsize=(10, 6))
cluster_colors = [POPULAR_RED, '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for c in range(5):
    mask = df['cluster'] == c
    label = cluster_profile.loc[cluster_profile['cluster']==c, 'dominant_persona'].values[0]
    ax.scatter(
        df.loc[mask, 'genre_enc'] + np.random.normal(0, 0.15, mask.sum()),
        df.loc[mask, 'condition_enc'] + np.random.normal(0, 0.1, mask.sum()),
        c=cluster_colors[c], alpha=0.5, s=40, label=f'Cluster {c}: {label}'
    )

ax.set_xlabel('Genre (encoded)', fontsize=12)
ax.set_ylabel('Condition (encoded: 0=Fair, 1=Good, 2=Worn)', fontsize=10)
ax.set_title('K-Means Clusters: Genre vs Condition', fontweight='bold', fontsize=14)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('clusters.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Random Forest Classifier (Supervised)
Unlike K-Means, the Random Forest *does* get the persona labels during training. It learns which combinations of genre, condition, month, and outlet map to which persona. Then we test it on data it hasn't seen to check if it actually learned the pattern.

This is the model that would run in production — a customer trades in a book, the system predicts their persona and triggers the right promo.


In [ ]:
# Train/test split — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing samples  : {len(X_test)}')
print(f'\nTraining class distribution:')
for enc, name in enumerate(le_persona.classes_):
    count = (y_train == enc).sum()
    print(f'  {name}: {count} samples')

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    class_weight='balanced'
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.2%}')
print()
print('Classification Report:')
print(classification_report(
    y_test, y_pred,
    target_names=le_persona.classes_
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Reds',
    xticklabels=le_persona.classes_,
    yticklabels=le_persona.classes_,
    ax=ax
)
ax.set_title('Confusion Matrix — Persona Prediction', fontweight='bold', fontsize=13)
ax.set_xlabel('Predicted Persona', fontsize=11)
ax.set_ylabel('Actual Persona', fontsize=11)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color=POPULAR_RED, alpha=0.85)
ax.set_title('Feature Importance — What Predicts Persona?', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nInsight: Genre is the strongest signal for identifying a customer persona.')

---
## 7. Persona Cards
Here we map each persona to a concrete marketing action and inventory recommendation. This is what Popular's staff would actually see.


In [ ]:
persona_cards = {
    'Parent': {
        'emoji': '👨‍👩‍👧',
        'description': 'Age 35-45. Trades in children\'s books and assessment books after school year ends. Good condition — child barely used them.',
        'signal': 'Children\'s/Assessment genre, Good condition, Jan or Nov/Dec',
        'action': 'Send next-level booklist in December. Offer EduPass family plan at heartland outlets.',
        'inventory': 'Stock more Children\'s and Assessment books at suburban outlets in Jan.',
        'color': POPULAR_RED,
        'size': df[df['persona']=='Parent'].shape[0]
    },
    'Student': {
        'emoji': '🎒',
        'description': 'Age 13-19. Trades in worn assessment books right after exams. Comes in to fund next semester\'s books.',
        'signal': 'Assessment genre, Fair/Worn condition, Jun or Nov/Dec',
        'action': 'Promote trade-in credit toward EduPass subscription. Send exam prep bundles in April and September.',
        'inventory': 'Stock more Assessment books near school-adjacent outlets before exam season.',
        'color': '#3498db',
        'size': df[df['persona']=='Student'].shape[0]
    },
    'Bookworm': {
        'emoji': '📖',
        'description': 'Age 20-35. Trades in fiction and manga year-round in good condition. Reads for pleasure, not obligation.',
        'signal': 'Fiction/Manga genre, Good condition, any month',
        'action': 'Recommend new fiction arrivals. Invite to BookFest. Offer curated reading lists.',
        'inventory': 'Expand Fiction and Manga sections at Bras Basah, Bugis, Tiong Bahru.',
        'color': '#2ecc71',
        'size': df[df['persona']=='Bookworm'].shape[0]
    },
    'Self-Improver': {
        'emoji': '💼',
        'description': 'Age 25-40. Trades in self-help and business books. Bought with good intentions, never finished. Working adult.',
        'signal': 'Self-Help/Business genre, Good condition, city outlets',
        'action': 'Recommend follow-up titles in the same topic. Promote weekend workshops or online courses.',
        'inventory': 'Expand Self-Help and Business sections at Plaza Singapura, Paya Lebar, VivoCity.',
        'color': '#f39c12',
        'size': df[df['persona']=='Self-Improver'].shape[0]
    },
    'Budget Shopper': {
        'emoji': '💰',
        'description': 'Any age. Trades in Fair/Worn books across any genre. Price-driven — comes in to maximise credit toward next purchase.',
        'signal': 'Any genre, Fair/Worn condition, any outlet, any month',
        'action': 'Highlight pre-loved section discounts. Offer credit top-up promotions.',
        'inventory': 'Ensure pre-loved section is well-stocked and visible at all outlets.',
        'color': '#9b59b6',
        'size': df[df['persona']=='Budget Shopper'].shape[0]
    },
}

print(f"{'Persona':<18} {'Size':>5}  {'Signal':<45} {'Action'}")
print('-' * 130)
for name, card in persona_cards.items():
    print(f"{card['emoji']} {name:<16} {card['size']:>5}  {card['signal']:<45} {card['action'][:60]}...")

In [ ]:
# Persona size bar chart
fig, ax = plt.subplots(figsize=(10, 5))
names  = list(persona_cards.keys())
sizes  = [persona_cards[n]['size'] for n in names]
colors = [persona_cards[n]['color'] for n in names]
emojis = [persona_cards[n]['emoji'] for n in names]

bars = ax.bar([f"{e}\n{n}" for e, n in zip(emojis, names)], sizes, color=colors, alpha=0.85, width=0.6)

for bar, size in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{size}\n({size/N*100:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Customer Persona Distribution — Trade-In Analytics', fontweight='bold', fontsize=14)
ax.set_ylabel('Number of Transactions')
ax.set_ylim(0, max(sizes) * 1.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('personas.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Outlet Intelligence
Which genres get traded in at which outlets? If Tampines gets a lot of Assessment trade-ins, that tells Popular to stock more Assessment books there.


In [ ]:
# Which genres are being traded in at each outlet?
outlet_genre = df.groupby(['outlet', 'genre']).size().unstack(fill_value=0)

# Focus on top 8 outlets by volume for readability
top_outlets = df['outlet'].value_counts().head(8).index
outlet_genre_top = outlet_genre.loc[outlet_genre.index.isin(top_outlets)]

fig, ax = plt.subplots(figsize=(14, 6))
outlet_genre_top.plot(kind='bar', ax=ax, colormap='Set2', width=0.8)
ax.set_title('Genre Trade-Ins by Outlet — Inventory Signal', fontweight='bold', fontsize=14)
ax.set_xlabel('Outlet')
ax.set_ylabel('Number of Trade-Ins')
ax.legend(title='Genre', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('outlet_intelligence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: High trade-ins of a genre at an outlet = high demand for that genre there.')
print('Popular should stock more of what customers are already reading at each location.')

---
## 9. Demo — New Trade-In Prediction
Here's what it looks like when a new trade-in comes in. The model takes the 4 inputs and predicts the persona.


In [ ]:
def predict_persona(genre, condition, month, outlet):
    """
    Predict customer persona from a single trade-in transaction.
    Returns the predicted persona, confidence, and recommended action.
    """
    X_new = pd.DataFrame([{
        'genre_enc':     le_genre.transform([genre])[0],
        'condition_enc': le_condition.transform([condition])[0],
        'month':         month,
        'outlet_enc':    le_outlet.transform([outlet])[0]
    }])

    pred_enc     = rf.predict(X_new)[0]
    proba        = rf.predict_proba(X_new)[0]
    persona_name = le_persona.inverse_transform([pred_enc])[0]
    confidence   = proba[pred_enc]
    card         = persona_cards[persona_name]

    print(f'Trade-In  : {genre} | {condition} | Month {month} | {outlet}')
    print(f'Predicted : {card["emoji"]}  {persona_name} ({confidence:.0%} confidence)')
    print(f'Action    : {card["action"]}')
    print()

# Scenario 1
# A parent returns their child's books after the school year
print('Scenario 1: Children\'s book in Good condition at Tampines Mall in November')
print()
predict_persona("Children's", 'Good', 11, 'Tampines Mall')

# Scenario 2
# A reader trades in a novel at a city outlet
print('Scenario 2: Fiction book in Good condition at Bras Basah Complex in March')
print()
predict_persona('Fiction', 'Good', 3, 'Bras Basah Complex')

# Scenario 3
# A price-conscious customer trades in a worn book at a suburban mall
print('Scenario 3: Self-Help book in Worn condition at Jurong Point in July')
print()
predict_persona('Self-Help', 'Worn', 7, 'Jurong Point')


---
## 9b. Interactive Demo
Try it yourself — pick a genre, condition, month, and outlet, then hit Predict.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Dropdowns
genre_dd = widgets.Dropdown(
    options=sorted(df['genre'].unique()),
    value='Assessment',
    description='Genre:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
)
condition_dd = widgets.Dropdown(
    options=['Good', 'Fair', 'Worn'],
    value='Fair',
    description='Condition:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
)
month_dd = widgets.Dropdown(
    options=[(name, i+1) for i, name in enumerate(
        ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])],
    value=11,
    description='Month:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
)
outlet_dd = widgets.Dropdown(
    options=sorted(df['outlet'].unique()),
    value='Tampines Mall',
    description='Outlet:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
)

btn = widgets.Button(
    description='Predict Persona',
    button_style='danger',
    layout=widgets.Layout(width='300px', height='40px'),
    style={'font_weight': 'bold'}
)

output = widgets.Output()

def on_predict(b):
    output.clear_output()
    with output:
        genre = genre_dd.value
        condition = condition_dd.value
        month = month_dd.value
        outlet = outlet_dd.value

        X_new = pd.DataFrame([{
            'genre_enc':     le_genre.transform([genre])[0],
            'condition_enc': le_condition.transform([condition])[0],
            'month':         month,
            'outlet_enc':    le_outlet.transform([outlet])[0]
        }])

        pred_enc = rf.predict(X_new)[0]
        proba = rf.predict_proba(X_new)[0]
        persona_name = le_persona.inverse_transform([pred_enc])[0]
        confidence = proba[pred_enc]
        card = persona_cards[persona_name]

        # All persona probabilities
        proba_str = '  '.join(
            f'{le_persona.inverse_transform([i])[0]}: {p:.0%}'
            for i, p in enumerate(proba) if p > 0.05
        )

        display(HTML(f'''
        <div style="background:#f9f9f9; border-left:4px solid {card['color']};
                    padding:15px; margin:10px 0; border-radius:4px; max-width:500px;">
            <div style="font-size:24px; margin-bottom:5px;">
                {card['emoji']}  <strong>{persona_name}</strong>
                <span style="color:#888; font-size:14px;">({confidence:.0%} confidence)</span>
            </div>
            <div style="color:#555; font-size:13px; margin-bottom:10px;">{card['description']}</div>
            <div style="font-size:13px;"><strong>Recommended action:</strong> {card['action']}</div>
            <div style="font-size:13px; margin-top:5px;"><strong>Inventory signal:</strong> {card['inventory']}</div>
            <div style="font-size:11px; color:#999; margin-top:10px;">Probabilities: {proba_str}</div>
        </div>
        '''))

btn.on_click(on_predict)

display(widgets.VBox([
    genre_dd, condition_dd, month_dd, outlet_dd,
    widgets.HTML('<div style="height:5px;"></div>'),
    btn, output
], layout=widgets.Layout(padding='10px')))


---
## 10. Summary

| Model | What it does | Result |
|---|---|---|
| K-Means | Finds natural groupings without labels | 5 clusters that roughly match our personas |
| Random Forest | Predicts persona from trade-in data | 86.5% accuracy on test set |

**What this means for Popular:**
- They can identify who a customer is from a single trade-in — no survey, no login
- Genre + condition + month + outlet is enough to predict persona with decent accuracy
- Trade-in patterns at each outlet tell them what to stock where
- They can trigger personalised promos at the right time for the right person

**Limitations:**
- This data is simulated — real accuracy depends on actual trade-in volume
- A customer can fit more than one persona; the model just picks the most likely one
- Persona labels would need human validation before going live
- Must comply with PDPA for any real customer data
